In [2]:
import pandas as pd
import os 
usuario = os.getlogin()

In [17]:
personas = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Comando Florence Nightingale\Proyectos\78_transicion sistemas prod\derechohabiencia\ece_personas_derechohabiencia.parquet")

if 'curp_hash32' in personas.columns:
    personas = personas.drop_duplicates(subset='curp_hash32', keep='first').reset_index(drop=True)

In [16]:
hoja_nueva= pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Comando Florence Nightingale\Proyectos\78_transicion sistemas prod\derechohabiencia\ece_consolidado_clues_derechohabiencia.parquet")

In [18]:
hoja_nueva

,clues,nombre_de_la_unidad,entidad,fecha,tipo_atencion,modulo,atenciones_totales,atenciones_derechohabientes,porcentaje_derechohabientes
0,BCIMB000734,HOSPITAL GENERAL TIJUANA,BAJA CALIFORNIA,2026-08-31,egresos,egresos,2,2,100.00
1,BCIMB000734,HOSPITAL GENERAL TIJUANA,BAJA CALIFORNIA,2026-08-31,urgencias,urgencias,1,1,100.00
2,BCIMB000734,HOSPITAL GENERAL TIJUANA,BAJA CALIFORNIA,2026-09-04,urgencias,urgencias,6,0,0.00
3,BCIMB000734,HOSPITAL GENERAL TIJUANA,BAJA CALIFORNIA,2026-09-08,urgencias,urgencias,1,0,0.00
4,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-18,consulta general,moce,3,0,0.00
...,...,...,...,...,...,...,...,...,...
35977,YNIMB000036,HOSPITAL DE ALTA ESPECIALIDAD AGUSTÍN O´HORÁN,YUCATAN,2026-09-09,procedimientos qx,cirugias,23,1,4.35
35978,YNIMB000036,HOSPITAL DE ALTA ESPECIALIDAD AGUSTÍN O´HORÁN,YUCATAN,2026-09-09,urgencias,urgencias,207,11,5.31
35979,YNIMB000036,HOSPITAL DE ALTA ESPECIALIDAD AGUSTÍN O´HORÁN,YUCATAN,2026-09-10,consulta de especialidad,moce,103,30,29.13
35980,YNIMB000036,HOSPITAL DE ALTA ESPECIALIDAD AGUSTÍN O´HORÁN,YUCATAN,2026-09-10,consulta general,moce,10,3,30.00


In [19]:
personas

,clues,nombre_de_la_unidad,entidad,fecha,curp_hash32,derechohabiente,tipo_atencion,modulo
0,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-18,ed0b8961e6a1a8afc77be0394dc014c6,False,consulta general,moce
1,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-18,d4874bebb862da31e7b1725667fd4830,False,consulta general,moce
2,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-18,0567952cb2e7243c70a0965386e017ca,False,consulta general,moce
3,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-19,d885714b1b108bf679405e73ed63b4b5,True,consulta general,moce
4,BSIMB000520,UNEME HEMODIÁLISIS,BAJA CALIFORNIA SUR,2026-08-19,a4a0a69c1afce589eacf81c45a495258,False,consulta general,moce
...,...,...,...,...,...,...,...,...
298688,CSIMB003704,HOSPITAL BÁSICO COMUNITARIO LAS ROSAS,CHIAPAS,2026-08-29,5bc6be1471d514e846f228ebd6fb5357,False,procedimientos qx,cirugias
298689,CSIMB000460,HOSPITAL GENERAL MARÍA IGNACIA GANDULFO COMITAN,CHIAPAS,2026-09-08,ef6f7d7d544c4e766b67289d43222228,False,procedimientos qx,cirugias
298690,YNIMB000012,HOSPITAL REGIONAL DE ALTA ESPECIALIDAD DE LA P...,HRAES,2026-09-07,57a6dd95f57be870f29978ff79c1b49e,False,procedimientos qx,cirugias
298691,DFIMB001822,HOSPITAL GENERAL XOCO,CIUDAD DE MEXICO,2026-09-08,8f6e4a17e8edc4cc2558e1069a398bbd,False,procedimientos qx,cirugias


In [2]:
from pathlib import Path


directorio_bases = Path(
    fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Comando Florence Nightingale\Proyectos\78_transicion sistemas prod\informe\tabla_resumen"
)


prefijo_base = "base_clues_sistemas_completa_"
bases_disponibles = []
for ruta in directorio_bases.glob(f"{prefijo_base}*.parquet"):
    try:
        fecha = pd.Timestamp(ruta.stem.removeprefix(prefijo_base).replace("_", "-"))
    except ValueError:
        continue
    bases_disponibles.append((fecha, ruta))

bases_disponibles.sort(key=lambda base: base[0])
if not bases_disponibles:
    raise FileNotFoundError(f"No se encontraron bases fechadas en: {directorio_bases}")

fecha_corte_actual, informe_actual_path = bases_disponibles[-1]
bases_anteriores = [base for base in bases_disponibles if base[0] < fecha_corte_actual]
if not bases_anteriores:
    raise FileNotFoundError("No se encontro un corte anterior al mas reciente")
fecha_corte_anterior, informe_corte_anterior_path = bases_anteriores[-1]

base_mas_reciente_path = directorio_bases / 'base_clues_sistemas_completa.parquet'
resumenes_disponibles = sorted(directorio_bases.glob('tabla_resumen_sistemas_????-??-??.xlsx'))
if base_mas_reciente_path.exists() and resumenes_disponibles:
    fecha_resumen_mas_reciente = pd.Timestamp(resumenes_disponibles[-1].stem.removeprefix('tabla_resumen_sistemas_'))
    if fecha_resumen_mas_reciente > fecha_corte_actual:
        fecha_corte_anterior, informe_corte_anterior_path = bases_disponibles[-1]
        fecha_corte_actual = fecha_resumen_mas_reciente
        informe_actual_path = base_mas_reciente_path


informe = pd.read_parquet(informe_actual_path)

print(f"Corte actual: {fecha_corte_actual:%Y-%m-%d}")
print(f"Corte anterior: {fecha_corte_anterior:%Y-%m-%d}")

Corte actual: 2026-09-09
Corte anterior: 2026-09-02


In [3]:
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [4]:
clues.columns

Index(['clues_imb', 'clues_ssa_y_sme', 'categoria_gerencial_uas',
       'categoria_gerencial', 'categoria_gerencial_ampliada',
       'categoria_gerencial_nueva', 'clave_de_la_entidad', 'entidad',
       'clave_del_municipio', 'municipio', 'localidad',
       'clave_de_la_localidad', 'nombre_de_la_unidad', 'nombre_comercial',
       'estatus_de_operacion', 'nivel_atencion', 'clave_de_tipologia',
       'nombre_de_tipologia', 'clave_de_subtipologia',
       'nombre_de_subtipologia', 'estrato_unidad', 'cve_ro', 'nombre_region',
       'latitud', 'longitud', 'organ_ro', 'FECHA DE INICIO DE OPERACION',
       'tipo_hbc'],
      dtype='object')

In [5]:
informe = informe.rename(columns={"clues": "clues_imb"})

In [6]:
hab_merge = (
    clues[["clues_imb", 'entidad', 'nombre_de_la_unidad']]

)
informe= informe.merge(hab_merge, on="clues_imb", how="left")

In [7]:
informe

,clues_imb,pqx_reportados_ece,pqx_incorporados_ece,pqx_reportados_sinba,pqx_totales,reporta_pqx_ece,reporta_pqx_sinba,consultas_reportadas_ece,consultas_incorporadas_ece,consultas_reportadas_sinba,...,reporta_consulta_ece,reporta_consulta_sinba,egresos_reportados_ece,egresos_incorporados_ece,egresos_reportados_sinba,egresos_totales,reporta_egreso_ece,reporta_egreso_sinba,entidad,nombre_de_la_unidad
0,BCIMB000010,0.0,0.0,2844.0,2844.0,False,True,0.0,0.0,35316.0,...,False,True,0.0,0.0,4711.0,4711.0,False,True,BAJA CALIFORNIA,HOSPITAL GENERAL DE ENSENADA
1,BCIMB000355,0.0,0.0,1179.0,1179.0,False,True,0.0,0.0,8209.0,...,False,True,0.0,0.0,1684.0,1684.0,False,True,BAJA CALIFORNIA,HOSPITAL GENERAL DE MEXICALI
2,BCIMB000623,0.0,0.0,0.0,0.0,False,False,0.0,0.0,15358.0,...,False,True,0.0,0.0,0.0,0.0,False,False,BAJA CALIFORNIA,HOSPITAL COMUNITARIO SAN FELIPE
3,BCIMB000676,0.0,0.0,1130.0,1130.0,False,True,0.0,0.0,2809.0,...,False,True,0.0,0.0,1044.0,1044.0,False,True,BAJA CALIFORNIA,HOSPITAL GENERAL DE TECATE
4,BCIMB000734,0.0,0.0,3806.0,3806.0,False,True,0.0,0.0,37589.0,...,False,True,2.0,2.0,8112.0,8114.0,True,True,BAJA CALIFORNIA,HOSPITAL GENERAL TIJUANA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,ZSIMB002015,0.0,0.0,271.0,271.0,False,True,0.0,0.0,10083.0,...,False,True,0.0,0.0,334.0,334.0,False,True,ZACATECAS,HOSPITAL COMUNITARIO SOMBRERETE
587,ZSIMB002126,0.0,0.0,4905.0,4905.0,False,True,0.0,0.0,10345.0,...,False,True,0.0,0.0,5015.0,5015.0,False,True,ZACATECAS,HOSPITAL DE LA MUJER
588,ZSIMB002213,0.0,0.0,0.0,0.0,False,False,0.0,0.0,12480.0,...,False,True,0.0,0.0,337.0,337.0,False,True,ZACATECAS,HOSPITAL DE ESPECIALIDADES DE SALUD MENTAL
589,ZSIMB002295,0.0,0.0,735.0,735.0,False,True,0.0,0.0,10470.0,...,False,True,0.0,0.0,1134.0,1134.0,False,True,ZACATECAS,HOSPITAL GENERAL LORETO


In [8]:
# Infraestructura: resumen para cards por CLUES (Consultas, Procedimientos Quirurgicos y Egresos)

import json

from pathlib import Path



base_output = Path.cwd() / 'reporte-_angeles' / 'public'

if not base_output.exists():

    # Fallback para ejecuciones fuera del workspace actual

    base_output = Path(r"C:\Users\jose.valdez\Downloads\reporteador\reporte-_angeles\public")



informe_corte_anterior = None

if informe_corte_anterior_path.exists():

    informe_corte_anterior = pd.read_parquet(informe_corte_anterior_path)

    if 'clues' in informe_corte_anterior.columns and 'clues_imb' not in informe_corte_anterior.columns:

        informe_corte_anterior = informe_corte_anterior.rename(columns={'clues': 'clues_imb'})



    hab_merge_cols = ['clues_imb', 'entidad', 'nombre_de_la_unidad']

    if all(col in clues.columns for col in hab_merge_cols):

        informe_corte_anterior = informe_corte_anterior.merge(

            clues[hab_merge_cols],

            on='clues_imb',

            how='left',

        )

else:

    print(f"[AVISO] No se encontro parquet del corte anterior: {informe_corte_anterior_path}")



def to_bool(s: pd.Series) -> pd.Series:

    if s.dtype == bool:

        return s.fillna(False)

    txt = s.astype(str).str.strip().str.lower()

    return txt.isin(['true', '1', 'si', 'yes'])



def to_number(s: pd.Series) -> pd.Series:

    # 0.0 se interpreta como cero; 500.0 se conserva como valor positivo

    txt = s.astype(str).str.strip()

    txt = txt.str.replace(' ', '', regex=False)

    txt = txt.str.replace(',', '.', regex=False)

    return pd.to_numeric(txt, errors='coerce').fillna(0)



def first_non_empty(series: pd.Series) -> str:

    for val in series:

        txt = str(val).strip()

        if txt and txt.lower() != 'nan':

            return txt

    return ''



def to_int(value: object) -> int:

    try:

        return int(float(value))

    except Exception:

        return 0



def build_cards_payload(

    df: pd.DataFrame,

    indicador: str,

    cols_num: list[str],

    col_bool_ece: str,

    col_bool_sinba: str,

    resumen_cols: dict | None = None,

    previous_payload: dict | None = None,

    previous_df: pd.DataFrame | None = None,

) -> dict:

    required_cols = ['clues_imb', *cols_num, col_bool_ece, col_bool_sinba]

    if resumen_cols:

        required_cols.extend([resumen_cols['ece'], resumen_cols['sinba'], resumen_cols['total']])

    # Evita columnas duplicadas (ej. total y ece usando la misma columna)

    required_cols = list(dict.fromkeys(required_cols))



    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:

        raise ValueError(f"[{indicador}] Faltan columnas en informe: {missing_cols}")



    tmp = df[required_cols].copy()

    tmp['entidad'] = df['entidad'] if 'entidad' in df.columns else ''

    tmp['nombre_de_la_unidad'] = df['nombre_de_la_unidad'] if 'nombre_de_la_unidad' in df.columns else ''



    tmp['ece_bool'] = to_bool(tmp[col_bool_ece])

    tmp['sinba_bool'] = to_bool(tmp[col_bool_sinba])



    for col in cols_num:

        tmp[f'{col}_num'] = to_number(tmp[col])



    if resumen_cols:

        tmp['resumen_ece_num'] = to_number(tmp[resumen_cols['ece']])

        tmp['resumen_sinba_num'] = to_number(tmp[resumen_cols['sinba']])

        tmp['resumen_total_num'] = to_number(tmp[resumen_cols['total']])



    agg_cols = {

        'ece_bool': 'max',

        'sinba_bool': 'max',

        'entidad': first_non_empty,

        'nombre_de_la_unidad': first_non_empty,

        **{f'{col}_num': 'max' for col in cols_num},

    }

    if resumen_cols:

        agg_cols.update({

            'resumen_ece_num': 'max',

            'resumen_sinba_num': 'max',

            'resumen_total_num': 'max',

        })



    by_clues = tmp.groupby('clues_imb', as_index=False).agg(agg_cols).rename(columns={

        'ece_bool': 'reporta_ece',

        'sinba_bool': 'reporta_sinba',

    })



    zeros_mask = (by_clues[[f'{col}_num' for col in cols_num]] == 0).all(axis=1)

    by_clues['cuatro_columnas_en_cero'] = zeros_mask



    # Calcular deltas por CLUES comparando contra corte anterior

    if previous_df is not None:

        prev_tmp = previous_df[required_cols].copy()

        prev_tmp['ece_bool'] = to_bool(prev_tmp[col_bool_ece])

        prev_tmp['sinba_bool'] = to_bool(prev_tmp[col_bool_sinba])



        prev_by_clues = prev_tmp.groupby('clues_imb', as_index=False).agg({

            'ece_bool': 'max',

            'sinba_bool': 'max',

        }).rename(columns={

            'ece_bool': 'prev_reporta_ece',

            'sinba_bool': 'prev_reporta_sinba',

        })



        by_clues = by_clues.merge(prev_by_clues, on='clues_imb', how='left')

        by_clues['prev_reporta_ece'] = by_clues['prev_reporta_ece'].fillna(False)

        by_clues['prev_reporta_sinba'] = by_clues['prev_reporta_sinba'].fillna(False)

    else:

        by_clues['prev_reporta_ece'] = False

        by_clues['prev_reporta_sinba'] = False



    only_ece = int(((by_clues['reporta_ece']) & (~by_clues['reporta_sinba'])).sum())

    both = int(((by_clues['reporta_ece']) & (by_clues['reporta_sinba'])).sum())

    only_sinba = int(((~by_clues['reporta_ece']) & (by_clues['reporta_sinba'])).sum())

    both_false = int(((~by_clues['reporta_ece']) & (~by_clues['reporta_sinba'])).sum())



    clues_cuatro_columnas_en_cero = int(zeros_mask.sum())

    total_clues_imb = int(by_clues['clues_imb'].nunique())

    total_clues_evaluadas = int(total_clues_imb - clues_cuatro_columnas_en_cero)

    no_reportaron = int(total_clues_imb - both_false)



    detalle_cols = [

        'clues_imb',

        'nombre_de_la_unidad',

        'entidad',

        'reporta_ece',

        'reporta_sinba',

        'cuatro_columnas_en_cero',

    ]

    if resumen_cols:

        detalle_cols.extend(['resumen_ece_num', 'resumen_sinba_num', 'resumen_total_num'])



    detalle_clues = by_clues[detalle_cols].copy()

    detalle_clues = detalle_clues.rename(columns={

        'reporta_ece': 'reporta_ece_bool',

        'reporta_sinba': 'reporta_sinba_bool',

        'resumen_ece_num': 'resumen_ece',

        'resumen_sinba_num': 'resumen_sinba',

        'resumen_total_num': 'resumen_total',

    })



    # Calcular delta_clues por cada CLUES

    detalle_clues['delta_clues_ece'] = 0

    detalle_clues['delta_clues_sinba'] = 0

    detalle_clues['delta_clues_ambas'] = 0



    for idx, row in by_clues.iterrows():

        prev_ece = to_int(row.get('prev_reporta_ece', False))

        prev_sinba = to_int(row.get('prev_reporta_sinba', False))



        curr_ece = to_int(row['reporta_ece'])

        curr_sinba = to_int(row['reporta_sinba'])



        # ece_bool: solo ECE (no SINBA)

        detalle_clues.at[idx, 'delta_clues_ece'] = (

            1 if (curr_ece and not curr_sinba) else 0

        ) - (

            1 if (prev_ece and not prev_sinba) else 0

        )



        # sinba_bool: solo SINBA (no ECE)

        detalle_clues.at[idx, 'delta_clues_sinba'] = (

            1 if (curr_sinba and not curr_ece) else 0

        ) - (

            1 if (prev_sinba and not prev_ece) else 0

        )



        # ambas: reporta ambos sistemas

        detalle_clues.at[idx, 'delta_clues_ambas'] = (

            1 if (curr_ece and curr_sinba) else 0

        ) - (

            1 if (prev_ece and prev_sinba) else 0

        )



    resumen_sistema = {

        'ece': int(by_clues['resumen_ece_num'].sum()) if resumen_cols else 0,

        'sinba': int(by_clues['resumen_sinba_num'].sum()) if resumen_cols else 0,

        'total': int(by_clues['resumen_total_num'].sum()) if resumen_cols else 0,

    }



    prev_cards = (previous_payload or {}).get('cards_clues', {})

    has_prev_cards = isinstance(prev_cards, dict) and len(prev_cards) > 0

    if has_prev_cards:

        delta_cards_clues = {

            'unicamente_en_ece': only_ece - to_int(prev_cards.get('unicamente_en_ece', only_ece)),

            'ambos_sistemas': both - to_int(prev_cards.get('ambos_sistemas', both)),

            'unicamente_en_sinba': only_sinba - to_int(prev_cards.get('unicamente_en_sinba', only_sinba)),

        }

    else:

        delta_cards_clues = {

            'unicamente_en_ece': 0,

            'ambos_sistemas': 0,

            'unicamente_en_sinba': 0,

        }



    prev_resumen = ((previous_payload or {}).get('metadata', {}) or {}).get('resumen_sistema', {})

    has_prev_resumen = isinstance(prev_resumen, dict) and len(prev_resumen) > 0

    if has_prev_resumen:

        delta_resumen_sistema = {

            'ece': resumen_sistema['ece'] - to_int(prev_resumen.get('ece', resumen_sistema['ece'])),

            'sinba': resumen_sistema['sinba'] - to_int(prev_resumen.get('sinba', resumen_sistema['sinba'])),

            'total': resumen_sistema['total'] - to_int(prev_resumen.get('total', resumen_sistema['total'])),

        }

    else:

        delta_resumen_sistema = {

            'ece': 0,

            'sinba': 0,

            'total': 0,

        }



    return {

        'indicador': indicador,

        'cards_clues': {

            'unicamente_en_ece': only_ece,

            'ambos_sistemas': both,

            'unicamente_en_sinba': only_sinba,

        },

        'metadata': {

            'total_clues_imb': total_clues_imb,

            'total_clues_evaluadas': total_clues_evaluadas,

            'clues_con_cuatro_columnas_en_cero': clues_cuatro_columnas_en_cero,

            'clues_sin_reporte_ambos': both_false,

            'no_reportaron': no_reportaron,

            'resumen_sistema': resumen_sistema,

            'delta_cards_clues': delta_cards_clues,

            'delta_resumen_sistema': delta_resumen_sistema,

            'columnas_numericas_fuente': cols_num,

            'columnas_fuente': required_cols,

        },

        'detalle_clues': detalle_clues.to_dict(orient='records'),

    }



def build_previous_payload(

    previous_df: pd.DataFrame | None,

    indicador: str,

    cols_num: list[str],

    col_bool_ece: str,

    col_bool_sinba: str,

    resumen_cols: dict | None = None,

) -> dict | None:

    if previous_df is None:

        return None

    try:

        return build_cards_payload(

            df=previous_df,

            indicador=indicador,

            cols_num=cols_num,

            col_bool_ece=col_bool_ece,

            col_bool_sinba=col_bool_sinba,

            resumen_cols=resumen_cols,

            previous_payload=None,

            previous_df=None,

        )

    except Exception as exc:

        print(f"[AVISO] No se pudo construir referencia historica para {indicador}: {exc}")

        return None



consultas_cards = build_cards_payload(

    df=informe,

    indicador='consultas',

    cols_num=[

        'consultas_reportadas_ece',

        'consultas_incorporadas_ece',

        'consultas_reportadas_sinba',

        'consultas_totales',

    ],

    col_bool_ece='reporta_consulta_ece',

    col_bool_sinba='reporta_consulta_sinba',

    resumen_cols={

        'ece': 'consultas_reportadas_ece',

        'sinba': 'consultas_reportadas_sinba',

        'total': 'consultas_totales',

    },

    previous_payload=build_previous_payload(

        previous_df=informe_corte_anterior,

        indicador='consultas',

        cols_num=[

            'consultas_reportadas_ece',

            'consultas_incorporadas_ece',

            'consultas_reportadas_sinba',

            'consultas_totales',

        ],

        col_bool_ece='reporta_consulta_ece',

        col_bool_sinba='reporta_consulta_sinba',

        resumen_cols={

            'ece': 'consultas_reportadas_ece',

            'sinba': 'consultas_reportadas_sinba',

            'total': 'consultas_totales',

        },

    ),

    previous_df=informe_corte_anterior,

)



pqx_cards = build_cards_payload(

    df=informe,

    indicador='procedimientos_quirurgicos',

    cols_num=[

        'pqx_reportados_ece',

        'pqx_incorporados_ece',

        'pqx_reportados_sinba',

        'pqx_totales',

    ],

    col_bool_ece='reporta_pqx_ece',

    col_bool_sinba='reporta_pqx_sinba',

    resumen_cols={

        'ece': 'pqx_reportados_ece',

        'sinba': 'pqx_reportados_sinba',

        'total': 'pqx_totales',

    },

    previous_payload=build_previous_payload(

        previous_df=informe_corte_anterior,

        indicador='procedimientos_quirurgicos',

        cols_num=[

            'pqx_reportados_ece',

            'pqx_incorporados_ece',

            'pqx_reportados_sinba',

            'pqx_totales',

        ],

        col_bool_ece='reporta_pqx_ece',

        col_bool_sinba='reporta_pqx_sinba',

        resumen_cols={

            'ece': 'pqx_reportados_ece',

            'sinba': 'pqx_reportados_sinba',

            'total': 'pqx_totales',

        },

    ),

    previous_df=informe_corte_anterior,

)



egresos_cards = build_cards_payload(

    df=informe,

    indicador='egresos_hospitalarios',

    cols_num=[

        'egresos_reportados_ece',

        'egresos_incorporados_ece',

        'egresos_reportados_sinba',

        'egresos_totales',

    ],

    col_bool_ece='reporta_egreso_ece',

    col_bool_sinba='reporta_egreso_sinba',

    resumen_cols={

        'ece': 'egresos_reportados_ece',

        'sinba': 'egresos_reportados_sinba',

        'total': 'egresos_totales',

    },

    previous_payload=build_previous_payload(

        previous_df=informe_corte_anterior,

        indicador='egresos_hospitalarios',

        cols_num=[

            'egresos_reportados_ece',

            'egresos_incorporados_ece',

            'egresos_reportados_sinba',

            'egresos_totales',

        ],

        col_bool_ece='reporta_egreso_ece',

        col_bool_sinba='reporta_egreso_sinba',

        resumen_cols={

            'ece': 'egresos_reportados_ece',

            'sinba': 'egresos_reportados_sinba',

            'total': 'egresos_totales',

        },

    ),

    previous_df=informe_corte_anterior,

)



infraestructura_cards = {

    'indicador': 'infraestructura_cards',

    'consultas': consultas_cards,

    'procedimientos_quirurgicos': pqx_cards,

    'egresos_hospitalarios': egresos_cards,

}

In [9]:
infraestructura_cards['fecha_corte'] = fecha_corte_actual.strftime('%Y-%m-%d')
infraestructura_cards['fecha_corte_anterior'] = fecha_corte_anterior.strftime('%Y-%m-%d')

In [10]:
import json
from pathlib import Path

base_output = Path.cwd() / 'reporte-_angeles' / 'public'
if not base_output.exists():
    # Fallback para ejecuciones fuera del workspace actual
    base_output = Path(r"C:\Users\jose.valdez\Downloads\reporteador\reporte-_angeles\public")
base_output.mkdir(parents=True, exist_ok=True)

infra_output_path = base_output / 'infraestructura_cards.json'
with infra_output_path.open('w', encoding='utf-8') as f:
    json.dump(infraestructura_cards, f, ensure_ascii=False, indent=2)

# Compatibilidad con la conexion previa de Procedimientos Quirurgicos
pqx_output_path = base_output / 'procedimientos_quirurgicos_cards.json'
with pqx_output_path.open('w', encoding='utf-8') as f:
    json.dump(pqx_cards, f, ensure_ascii=False, indent=2)

# Historico semanal en un solo parquet (una fila por indicador y semana)
hist_dir = base_output / 'historico'
hist_dir.mkdir(parents=True, exist_ok=True)
hist_path = hist_dir / 'infraestructura_hist.parquet'

fecha_corte = fecha_corte_actual.normalize()
semana_inicio = fecha_corte - pd.Timedelta(days=fecha_corte.weekday())
semana_fin = semana_inicio + pd.Timedelta(days=6)

def to_int(value: object) -> int:
    try:
        return int(float(value))
    except Exception:
        return 0

rows = []
for indicador_key in ['consultas', 'procedimientos_quirurgicos', 'egresos_hospitalarios']:
    payload = infraestructura_cards.get(indicador_key, {}) or {}
    cards = payload.get('cards_clues', {}) or {}
    meta = payload.get('metadata', {}) or {}
    resumen = meta.get('resumen_sistema', {}) or {}
    delta_cards = meta.get('delta_cards_clues', {}) or {}
    delta_resumen = meta.get('delta_resumen_sistema', {}) or {}

    rows.append({
        'fecha_corte': fecha_corte,
        'semana_inicio': semana_inicio,
        'semana_fin': semana_fin,
        'indicador': indicador_key,
        'unicamente_en_ece': to_int(cards.get('unicamente_en_ece', 0)),
        'ambos_sistemas': to_int(cards.get('ambos_sistemas', 0)),
        'unicamente_en_sinba': to_int(cards.get('unicamente_en_sinba', 0)),
        'total_clues_evaluadas': to_int(meta.get('total_clues_evaluadas', 0)),
        'no_reportaron': to_int(meta.get('no_reportaron', 0)),
        'resumen_ece': to_int(resumen.get('ece', 0)),
        'resumen_sinba': to_int(resumen.get('sinba', 0)),
        'resumen_total': to_int(resumen.get('total', 0)),
        'delta_unicamente_en_ece': to_int(delta_cards.get('unicamente_en_ece', 0)),
        'delta_ambos_sistemas': to_int(delta_cards.get('ambos_sistemas', 0)),
        'delta_unicamente_en_sinba': to_int(delta_cards.get('unicamente_en_sinba', 0)),
        'delta_resumen_ece': to_int(delta_resumen.get('ece', 0)),
        'delta_resumen_sinba': to_int(delta_resumen.get('sinba', 0)),
        'delta_resumen_total': to_int(delta_resumen.get('total', 0)),
    })

new_week_df = pd.DataFrame(rows)

if hist_path.exists():
    hist_df = pd.read_parquet(hist_path)
    if not hist_df.empty:
        hist_df['fecha_corte'] = pd.to_datetime(hist_df['fecha_corte']).dt.normalize()
        hist_df = hist_df[hist_df['fecha_corte'] <= fecha_corte]

        if 'semana_inicio' in hist_df.columns:
            hist_df['semana_inicio'] = pd.to_datetime(hist_df['semana_inicio']).dt.normalize()
        else:
            hist_df['semana_inicio'] = hist_df['fecha_corte'] - pd.to_timedelta(hist_df['fecha_corte'].dt.weekday, unit='D')

        if 'semana_fin' in hist_df.columns:
            hist_df['semana_fin'] = pd.to_datetime(hist_df['semana_fin']).dt.normalize()
        else:
            hist_df['semana_fin'] = hist_df['semana_inicio'] + pd.Timedelta(days=6)

        # Si se corre varias veces en la misma semana, sustituye por el corte mas reciente.
        hist_df = hist_df[hist_df['semana_inicio'] != semana_inicio]

    out_df = pd.concat([hist_df, new_week_df], ignore_index=True)
else:
    out_df = new_week_df.copy()

# Garantiza una sola fila por indicador por semana, conservando la mas reciente.
out_df = out_df.sort_values(['semana_inicio', 'indicador', 'fecha_corte']).drop_duplicates(
    subset=['semana_inicio', 'indicador'],
    keep='last'
).reset_index(drop=True)

out_df.to_parquet(hist_path, index=False)

infra_output_path, pqx_output_path, hist_path, out_df.tail(3)

(WindowsPath('C:/Users/jose.valdez/Downloads/inf-ece/informe-ece/reporte-_angeles/public/infraestructura_cards.json'),
 WindowsPath('C:/Users/jose.valdez/Downloads/inf-ece/informe-ece/reporte-_angeles/public/procedimientos_quirurgicos_cards.json'),
 WindowsPath('C:/Users/jose.valdez/Downloads/inf-ece/informe-ece/reporte-_angeles/public/historico/infraestructura_hist.parquet'),
    fecha_corte                   indicador  unicamente_en_ece  ambos_sistemas  \
 12  2026-09-09                   consultas                 10              91   
 13  2026-09-09       egresos_hospitalarios                 10              67   
 14  2026-09-09  procedimientos_quirurgicos                 11              44   
 
     unicamente_en_sinba  total_clues_evaluadas  no_reportaron  resumen_ece  \
 12                  477                    588            578       329571   
 13                  453                    585            530        57261   
 14                  398                    538      